In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv

# Setup

In [32]:
from pathlib import Path
import pandas as pd

data_path = Path('C:/Users/alfon/Desktop/UN/Data/Metadata Comparison')

metadata_file_1_path = data_path / 'Metadata_Automation_index_7_2025.xlsx'
metadata_file_2_path = data_path / 'Metadata_Automation_index_7_2026.xlsx'

overall_summary_sheet = 'Summary Stats'
kpi_summary_sheet = 'KPI Data Summary'
country_summary_sheet = 'Summary Stats by Country'
metadata_sheet = 'Metadata_automated'

sheet_names = [overall_summary_sheet, kpi_summary_sheet, country_summary_sheet]

metadata_1_name = 'index_7_2025'
metadata_2_name = 'index_7_2026'

escwa_members = ['DZA','BHR','COM','DJI','EGY','IRQ','JOR','KWT','LBN','LBY','MRT',
                 'MAR','OMN','PSE','QAT','SAU','SOM','SDN','SYR','TUN','ARE','YEM']

# ISO -> Country lookup from the first two columns (Country, ISO) of the metadata sheet, both files combined
iso_country = pd.concat([
    pd.read_excel(p, sheet_name=metadata_sheet, usecols=[0, 1])
    for p in (metadata_file_1_path, metadata_file_2_path)
]).drop_duplicates()
iso_to_country = dict(zip(iso_country['ISO'], iso_country['Country']))

# ESCWA members that appear in the data
escwa_iso_to_country = {iso: iso_to_country[iso] for iso in escwa_members if iso in iso_to_country}
escwa_missing = [iso for iso in escwa_members if iso not in iso_to_country]
print(f"ESCWA members in data: {len(escwa_iso_to_country)}/{len(escwa_members)}")
if escwa_missing:
    print(f"   Not in data: {escwa_missing}")


# Read each sheet of each file into its own DataFrame
metadata_1 = pd.read_excel(metadata_file_1_path, sheet_name=sheet_names)
metadata_2 = pd.read_excel(metadata_file_2_path, sheet_name=sheet_names)

# ---------- Structure checks (run on the ORIGINAL column names) ----------
n1, n2 = metadata_1_name, metadata_2_name

for name in sheet_names:
    df1 = metadata_1[name]
    df2 = metadata_2[name]

    print(f"\n=== {name} ===")
    print(f"Shape: {n1}={df1.shape}, {n2}={df2.shape}")

    # 1. Column count
    if df1.shape[1] == df2.shape[1]:
        print(f"✅ Same number of columns ({df1.shape[1]})")
    else:
        print(f"⚠️ Column count differs: {n1}={df1.shape[1]}, {n2}={df2.shape[1]}")

    # 2. Column names (as sets, ignoring order)
    cols1, cols2 = set(df1.columns), set(df2.columns)
    if cols1 == cols2:
        print("✅ Same column names")
    else:
        print("⚠️ Column names differ")
        only_1 = sorted(cols1 - cols2, key=str)
        only_2 = sorted(cols2 - cols1, key=str)
        if only_1:
            print(f"   Only in {n1}: {only_1}")
        if only_2:
            print(f"   Only in {n2}: {only_2}")

    # 3. Column order (only meaningful if names match)
    if cols1 == cols2:
        if list(df1.columns) == list(df2.columns):
            print("✅ Same column order")
        else:
            print("⚠️ Same columns but different order")

    # 4. Data types for shared columns
    shared = [c for c in df1.columns if c in cols2]
    dtype_diffs = {
        c: (str(df1[c].dtype), str(df2[c].dtype))
        for c in shared
        if df1[c].dtype != df2[c].dtype
    }
    if not dtype_diffs:
        print("✅ Same dtypes on shared columns")
    else:
        print("⚠️ Dtype differences:")
        for c, (d1, d2) in dtype_diffs.items():
            print(f"   {c}: {n1}={d1}, {n2}={d2}")

    # 5. Row count
    if df1.shape[0] == df2.shape[0]:
        print(f"✅ Same number of rows ({df1.shape[0]})")
    else:
        print(f"⚠️ Row count differs: {n1}={df1.shape[0]}, {n2}={df2.shape[0]}")


# ---------- Add the metadata name to the column names ----------
def suffix_columns(df, suffix, keep=()):
    """Append '_<suffix>' to every column except those in `keep` (e.g. join keys)."""
    return df.rename(columns={c: f"{c}_{suffix}" for c in df.columns if c not in keep})

key_cols = []  # e.g. ['Country'] to leave join keys unsuffixed

metadata_1 = {s: suffix_columns(df, metadata_1_name, key_cols) for s, df in metadata_1.items()}
metadata_2 = {s: suffix_columns(df, metadata_2_name, key_cols) for s, df in metadata_2.items()}

# First file
overall_summary_df_1 = metadata_1[overall_summary_sheet]
kpi_summary_df_1 = metadata_1[kpi_summary_sheet]
country_summary_df_1 = metadata_1[country_summary_sheet]

# Second file
overall_summary_df_2 = metadata_2[overall_summary_sheet]
kpi_summary_df_2 = metadata_2[kpi_summary_sheet]
country_summary_df_2 = metadata_2[country_summary_sheet]

ESCWA members in data: 13/22
   Not in data: ['COM', 'DJI', 'IRQ', 'LBY', 'PSE', 'SOM', 'SDN', 'SYR', 'YEM']

=== Summary Stats ===
Shape: index_7_2025=(9, 3), index_7_2026=(9, 3)
✅ Same number of columns (3)
✅ Same column names
✅ Same column order
✅ Same dtypes on shared columns
✅ Same number of rows (9)

=== KPI Data Summary ===
Shape: index_7_2025=(78, 35), index_7_2026=(78, 35)
✅ Same number of columns (35)
✅ Same column names
✅ Same column order
⚠️ Dtype differences:
   year_min: index_7_2025=int64, index_7_2026=float64
   year_max: index_7_2025=int64, index_7_2026=float64
   year_range: index_7_2025=int64, index_7_2026=float64
   year_median: index_7_2025=int64, index_7_2026=float64
   most_common_year: index_7_2025=int64, index_7_2026=float64
✅ Same number of rows (78)

=== Summary Stats by Country ===
Shape: index_7_2025=(139, 18), index_7_2026=(139, 18)
✅ Same number of columns (18)
✅ Same column names
✅ Same column order
✅ Same dtypes on shared columns
✅ Same number of rows (

In [33]:
df1 = overall_summary_df_1.copy()
df2 = overall_summary_df_2.copy()

# Label column and value column (positional, since the names carry the suffixes now)
key1, val1 = df1.columns[0], df1.columns[1]   # <- adjust if needed
key2, val2 = df2.columns[0], df2.columns[1]   # <- adjust if needed

# Remove the total row(s)
def drop_total(df, key):
    return df[~df[key].astype(str).str.strip().str.lower().str.startswith('total')]

df1 = drop_total(df1, key1)
df2 = drop_total(df2, key2)

# Join side by side on the label, keeping the original row order
idx1 = df1[key1].tolist()
idx2 = [k for k in df2[key2].tolist() if k not in idx1]   # rows only in 2026 go at the end
order = idx1 + idx2

overall_summary_df = (
    df1.set_index(key1)
       .join(df2.set_index(key2), how='outer', sort=False)
       .reindex(order)
       .rename_axis('Metric')
       .reset_index()
)

# Percentage columns (positional: Category, Count, Percentage)
pct1 = df1.columns[2]   # Percentage_index_7_2025
pct2 = df2.columns[2]   # Percentage_index_7_2026

# Make sure the percentage columns are numeric
overall_summary_df[pct1] = pd.to_numeric(overall_summary_df[pct1], errors='coerce')
overall_summary_df[pct2] = pd.to_numeric(overall_summary_df[pct2], errors='coerce')

# Single new column: percentage-point change (2026 minus 2025)
overall_summary_df['pct_change_delta'] = (overall_summary_df[pct2] - overall_summary_df[pct1]).round(2)

# Rename the category labels (these are row values, not column headers)
col_renames = {
    'A_total'                         : 'Bucket A: Data timely and accurate',
    'A1_cap_adjusted_match'           : 'Bucket A1: Data timely and accurate but capped',
    'B_total'                         : 'Bucket B: Data old but accurate',
    'C_data_available_no_match'       : 'Bucket C: Historical data available but no match',
    'D_possible_impute'               : 'Bucket D: Expert judgement / imputed',
    'E_no_historical_data'            : 'Bucket E: No Historical data available (Report data available)',
    'F_no_report_data'                : 'Bucket F: No Report data available (Historical data available)',
    'G_no_historical_and_report_data' : 'Bucket G: Neither Historical nor Report data available',
}

overall_summary_df['Metric'] = overall_summary_df['Metric'].replace(col_renames)

overall_summary_df

,Metric,Count_index_7_2025,Percentage_index_7_2025,Count_index_7_2026,Percentage_index_7_2026,pct_change_delta
0,Bucket A: Data timely and accurate,6765,62.39,2879,26.23,-36.16
1,Bucket A1: Data timely and accurate but capped,0,0.00,0,0.00,0.00
2,Bucket B: Data old but accurate,627,5.78,3644,33.20,27.42
3,Bucket C: Historical data available but no match,1829,16.87,1757,16.01,-0.86
4,Bucket D: Expert judgement / imputed,65,0.60,1169,10.65,10.05
5,Bucket E: No Historical data available (Report...,346,3.19,315,2.87,-0.32
6,Bucket F: No Report data available (Historical...,847,7.81,397,3.62,-4.19
7,Bucket G: Neither Historical nor Report data a...,364,3.36,814,7.42,4.06


# KPI Comparison

In [34]:
n1, n2 = metadata_1_name, metadata_2_name

# Columns to keep from each file (names carry the suffixes now)
id1, name1, pct_a1 = f'KPI ID_{n1}', f'Series Name_{n1}', f'pct_A_{n1}'
id2, name2, pct_a2 = f'KPI ID_{n2}', f'Series Name_{n2}', f'pct_A_{n2}'

df1 = kpi_summary_df_1[[id1, name1, pct_a1]].copy()
df2 = kpi_summary_df_2[[id2, name2, pct_a2]].copy()

# Join side by side on KPI ID, keeping the original row order
idx1 = df1[id1].tolist()
idx2 = [k for k in df2[id2].tolist() if k not in idx1]   # KPIs only in 2026 go at the end
order = idx1 + idx2

kpi_summary_df = (
    df1.set_index(id1)
       .join(df2.set_index(id2), how='outer', sort=False)
       .reindex(order)
       .rename_axis('KPI ID')
       .reset_index()
)

# Single Series Name column (fall back to 2026 name for KPIs only in 2026)
kpi_summary_df.insert(1, 'Series Name', kpi_summary_df[name1].fillna(kpi_summary_df[name2]))
kpi_summary_df = kpi_summary_df.drop(columns=[name1, name2])

# Make sure the percentage columns are numeric
kpi_summary_df[pct_a1] = pd.to_numeric(kpi_summary_df[pct_a1], errors='coerce')
kpi_summary_df[pct_a2] = pd.to_numeric(kpi_summary_df[pct_a2], errors='coerce')

# Percentage-point change in Bucket A share (2026 minus 2025)
kpi_summary_df['pct_A_change_delta'] = (kpi_summary_df[pct_a2] - kpi_summary_df[pct_a1]).round(2)

kpi_summary_df

,KPI ID,Series Name,pct_A_index_7_2025,pct_A_index_7_2026,pct_A_change_delta
0,116,Operational stability for businesses,95.7,23.7,-72.0
1,117,Government effectiveness,95.7,47.5,-48.2
2,118,Regulatory Quality,95.7,48.2,-47.5
3,119,Rule of Law,95.7,56.1,-39.6
4,123,Expenditure on education,41.0,32.4,-8.6
...,...,...,...,...,...
73,3240,"Youth demographic dividend, %",95.7,69.1,-26.6
74,3241,University industry & international engagement...,74.1,40.3,-33.8
75,447,"Venture capital recipients, deals/bn PPP$ GDP",86.3,10.8,-75.5
76,448,Production and export complexity,89.9,48.2,-41.7


# Country Comparison (ESCWA members)

In [35]:
n1, n2 = metadata_1_name, metadata_2_name

# Country -> ISO, restricted to ESCWA members (inverse of escwa_iso_to_country)
escwa_country_to_iso = {country: iso for iso, country in escwa_iso_to_country.items()}

def a_match(df, suffix):
    """ESCWA rows only: ISO, Country and A_match_pct = A_total_pct + A1_cap_adjusted_match_pct (as %, 0-100)."""
    country = f'Country_{suffix}'
    out = df[df[country].isin(escwa_country_to_iso)].copy()
    out['ISO'] = out[country].map(escwa_country_to_iso)
    a_pct = pd.to_numeric(out[f'A_total_pct_{suffix}'], errors='coerce')
    a1_pct = pd.to_numeric(out[f'A1_cap_adjusted_match_pct_{suffix}'], errors='coerce')
    out[f'A_match_pct_{suffix}'] = ((a_pct + a1_pct) * 100).round(2)   # sheet stores fractions
    return out[['ISO', country, f'A_match_pct_{suffix}']]

df1 = a_match(country_summary_df_1, n1)
df2 = a_match(country_summary_df_2, n2)

# Join side by side on ISO, keeping the ESCWA list order
order = [iso for iso in escwa_members if iso in set(df1['ISO']) | set(df2['ISO'])]

country_summary_df = (
    df1.set_index('ISO')
       .join(df2.set_index('ISO'), how='outer', sort=False)
       .reindex(order)
       .rename_axis('ISO')
       .reset_index()
)

# Single Country column (fall back to 2026 name for countries only in 2026)
country_summary_df.insert(1, 'Country', country_summary_df[f'Country_{n1}'].fillna(country_summary_df[f'Country_{n2}']))
country_summary_df = country_summary_df.drop(columns=[f'Country_{n1}', f'Country_{n2}'])

# Percentage-point change in A match share (2026 minus 2025)
country_summary_df['A_match_pct_change_delta'] = (
    country_summary_df[f'A_match_pct_{n2}'] - country_summary_df[f'A_match_pct_{n1}']
).round(2)

country_summary_df

,ISO,Country,A_match_pct_index_7_2025,A_match_pct_index_7_2026,A_match_pct_change_delta
0,DZA,Algeria,69.0,24.0,-45.0
1,BHR,Bahrain,62.0,22.0,-40.0
2,EGY,Egypt,72.0,24.0,-48.0
3,JOR,Jordan,67.0,23.0,-44.0
4,KWT,Kuwait,63.0,25.0,-38.0
5,LBN,Lebanon,62.0,43.0,-19.0
6,MRT,Mauritania,55.0,37.0,-18.0
7,MAR,Morocco,64.0,27.0,-37.0
8,OMN,Oman,63.0,22.0,-41.0
9,QAT,Qatar,71.0,24.0,-47.0


# Write to Excel

In [36]:
from utils.excel_functions import write_dataframes_to_excel_v2

In [37]:
comparison_output_path = data_path / f'Metadata_Comparison_{metadata_1_name}_vs_{metadata_2_name}.xlsx'

write_dataframes_to_excel_v2(
    {
        'Overall Summary'    : overall_summary_df,
        'KPI Comparison'     : kpi_summary_df,
        'Country Comparison' : country_summary_df,
    },
    output_path=comparison_output_path,
    delta_cols={
        'Overall Summary'    : ['pct_change_delta'],
        'KPI Comparison'     : ['pct_A_change_delta'],
        'Country Comparison' : ['A_match_pct_change_delta'],
    },
    shade_cols={
        'Overall Summary'    : [[f'Count_{metadata_1_name}', f'Percentage_{metadata_1_name}'],
                                [f'Count_{metadata_2_name}', f'Percentage_{metadata_2_name}']],
        'KPI Comparison'     : [[f'pct_A_{metadata_1_name}'], [f'pct_A_{metadata_2_name}']],
        'Country Comparison' : [[f'A_match_pct_{metadata_1_name}'], [f'A_match_pct_{metadata_2_name}']],
    },
)

Saved: C:\Users\alfon\Desktop\UN\Data\Metadata Comparison\Metadata_Comparison_index_7_2025_vs_index_7_2026.xlsx
